In [ ]:
import os
import re
from collections import defaultdict
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
#result_dir = '/result_opt'
#csv_file = 'res.csv'

#result_dir = '/result_full_sys'
#csv_file = 'res_full_sys.csv'

#result_dir = '/result_gnr_full_sys_sweep'
#csv_file = 'res_full_sys_sweep.csv'

#result_dir = '/result_gnr_new_models_sweep'
#csv_file = 'res_new_models_sweep.csv'

#result_dir = '/result_gnr_full_sys_all_models_iter1'
#csv_file = 'result_gnr_full_sys_all_models_iter1.csv'

#result_dir = '/result_spr_all_models_sweep'
#csv_file = 'result_spr_all_models_sweep.csv'

#result_dir = '/result_v1_gnr_full_sys_sweep'
#csv_file = 'result_v1_gnr_full_sys_sweep.csv'

#result_dir = '/result_v1_spr_all_models_sweep'
#csv_file = 'result_v1_spr_full_sys_sweep.csv'
#small_csv = 'result_v1_spr_full_sys_sweep_small.csv'



In [ ]:
def get_json(fn):
    j = None
    try:
        with open(fn, 'r') as f:
            j = json.load(f)
    except Exception as e:
        print(f"Opening file {fn} failed with exception {e}")
        #sys.exit(1)
    return j

def get_result_files(directory):
    all_files = defaultdict(lambda: defaultdict(list))
    print(f"Getting result files from {directory}")
    for root, dirs, files in os.walk(directory):
        for file in files:
            #all_files.append(os.path.join(root, file))
            r = root[len(directory)+1:].split('/')

            if len(r) < 2:
                continue
            all_files[r[0]][r[1]].append(os.path.join(root, file))
            #all_files[(r[0], r[1])].append(file)
    return all_files

def get_highest_concurrency_file(files, sep):
    t = [(int(f[f.rfind('C')+1:f.rfind(sep)]), f) for f in files]
    return max(t)

def find_model_config(model_configs, model):
    m = model.replace('--', '/')
    for mc in model_configs:
        if m == mc['model']:
            return mc

def extract_results_from_json(fn):
    j = get_json(fn)
    return {
        'Model': j['model_id'],
        'Input Tokens': int(j['total_input_tokens']/j['num_prompts']),
        'Output Tokens': int(j['total_output_tokens']/j['num_prompts']),
        'Concurrency': int(j['request_rate']),
        'Output tput (tokens/sec)': round(j['output_throughput'], 2),
        'Total tput (tokens/sec)': round(j['total_token_throughput'], 2),
        'TTFT avg (ms)': round(j['mean_ttft_ms'], 2),
        'TTFT P50 (ms)': round(j['p50_ttft_ms'], 2),
        'TTFT P90 (ms)': round(j['p90_ttft_ms'], 2),
        'TTFT P99 (ms)': round(j['p99_ttft_ms'], 2),
        'TPOT avg (ms)': round(j['mean_tpot_ms'], 2),
        'TPOT P50 (ms)': round(j['p50_tpot_ms'], 2),
        'TPOT P90 (ms)': round(j['p90_tpot_ms'], 2),
        'TPOT P99 (ms)': round(j['p99_tpot_ms'], 2),
        'Request Throughput': round(j['request_throughput'], 2),
        'Request Latency avg (s)': round(j['mean_e2el_ms']/1000, 2),
        'Request Latency P50 (s)': round(j['p50_e2el_ms']/1000, 2),
        'Request Latency P90 (s)': round(j['p90_e2el_ms']/1000, 2),
        'Request Latency P99 (s)': round(j['p99_e2el_ms']/1000, 2),
    }

def extract_results_from_json_embed(model, fns, model_config):
    j = get_json(fns)
    mc = model_config['test_parameters']

    result = {
            'Model': model,
            'Input Tokens': int(j['total_input_tokens']/j['num_prompts']),
            'Concurrency': int(j['request_rate']),
            'Embedding dimension': mc["embedding_dimension"],
            'Num parameters': mc["num_parameters"],
            'Max tokens': mc["max_tokens"],
            'MTEB rank': mc["mteb_rank"],
            'Request Latency avg (ms)': round(j['mean_e2el_ms'], 2),
            'Request Latency P50 (ms)': round(j['p50_e2el_ms'], 2),
            'Request Latency P90 (ms)': round(j['p90_e2el_ms'], 2),
            'Request Latency P99 (ms)': round(j['p99_e2el_ms'], 2),
            'Num Requests': int(j['num_prompts']),
            'Throughput (req/sec)': round(j['request_throughput'], 2)
    }
    return result


In [ ]:
tag='result_quick_0514'
result_dir = os.path.join(os.getcwd(), 'results/spr/'+tag)
print(result_dir)
csv_file = os.path.join(result_dir, tag+'.csv')
small_csv = os.path.join(result_dir, tag+'_small.csv')

In [ ]:
files = get_result_files(result_dir)

In [ ]:
def parse_num_parameters(val):
    if isinstance(val, str):
        if val.endswith('B'):
            return float(val[:-1]) * 1000  # 1B -> 1000M
        if val.endswith('M'):
            return float(val[:-1])
    return float(val)



In [ ]:
values = []
confd = os.path.join(os.getcwd(), 'configs/spr/')
model_config_file = os.path.join(confd, 'models_embed.json')
model_config = get_json(model_config_file)
sep = '-'
for model, batches in files.items():
    mc = find_model_config(model_config, model)

    for fns in batches.values():
        #print(fns)
        _, fn = get_highest_concurrency_file(fns, sep)
        #print(fn, "***************************************************")
        values.append(extract_results_from_json_embed(model, fn, mc))
df = pd.DataFrame(values)
df['Num parameters (M)'] = df['Num parameters'].apply(parse_num_parameters)
df = df.sort_values(by=['Num parameters (M)',  'Input Tokens',  'Model', 'Concurrency'], ascending=[False, True, True, True])

df

In [ ]:
df = df.sort_values(by=['Input Tokens', 'Num parameters (M)',   'Model', 'Concurrency'], ascending=[True, False, True, True])
df

In [ ]:
import matplotlib.pyplot as plt

for tokens in [128, 256, 512]:
    df_subset = df[df['Input Tokens'] == tokens]
    plt.figure(figsize=(12, 6))
    plt.bar(df_subset['Model'], df_subset['Request Latency P90 (ms)'])
    plt.xlabel('Model')
    plt.ylabel('Request Latency P90 (ms)')
    plt.title(f'Request Latency P90 (ms) for Input Tokens = {tokens}')
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

In [ ]:
def param_group(num_m):
    if num_m >= 7000:
        return '7B+'
    elif num_m >= 1000:
        return '1B-7B'
    elif num_m >= 500:
        return '500M-1B'
    else:
        return '<500M'

df['Param Group'] = df['Num parameters (M)'].apply(param_group)

for tokens in [128, 256, 512]:
    df_subset = df[df['Input Tokens'] == tokens]
    plt.figure(figsize=(12, 6))
    for group in ['7B+', '1B-7B', '500M-1B', '<500M']:
        group_df = df_subset[df_subset['Param Group'] == group]
        plt.bar(group_df['Model'], group_df['Request Latency P90 (ms)'], label=group)
    plt.xlabel('Model')
    plt.ylabel('Request Latency P90 (ms)')
    plt.title(f'Request Latency P90 (ms) for Input Tokens = {tokens}')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Param Group')
    plt.yscale('log')  # Add this line for logarithmic y-axis
    plt.tight_layout()
    plt.show()

In [ ]:
df.to_csv(csv_file, index=False)
df[['Model', 'Input Tokens', 'Concurrency', 'Throughput (req/sec)', 'Request Latency P90 (ms)']].to_csv(small_csv, index=False)

In [ ]:
df[['Model', 'Input Tokens', 'Concurrency', 'Throughput (req/sec)', 'Request Latency P90 (ms)']]

In [ ]:
values = []
for batches in files.values():
    for fns in batches.values():
        #print(fns)
        _, fn = get_highest_concurrency_file(fns)
        #print(fn, "***************************************************")
        values.append(extract_results_from_json(fn))
df = pd.DataFrame(values)
df = df.sort_values(by=['Model', 'Input Tokens'], ascending=[True, True])
df

In [ ]:
df.to_csv(csv_file, index=False)
df[['Model', 'Input Tokens', 'Output Tokens', 'Concurrency', 'Output tput (tokens/sec)', 'TTFT P90 (ms)', 'TPOT P90 (ms)']].to_csv(small_csv, index=False)

In [ ]:
df[['Model', 'Input Tokens', 'Output Tokens', 'Concurrency', 'Output tput (tokens/sec)', 'TTFT P90 (ms)', 'TPOT P90 (ms)']]